# Librerías LLM en python
- **Scikit-Learn** el clásico de machine learning con todos los datos en una máquina
- **Mllib librería** para machine learning pensada para clúster con información repartida en varias máquinas

In [1]:
# Requiere winutils.exe en HADOOP_HOME\bin (necesario en Windows)
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop\winutils\hadoop-3.3.6"
os.environ["hadoop.home.dir"] = os.environ["HADOOP_HOME"]
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + ";" + os.environ["PATH"]

In [2]:
import pandas as pd
import numpy as np

print("Generando archivo CSV sintético de 50,000 transacciones (esto tomará unos segundos)...")
n_rows = 50000
np.random.seed(42) # Para reproducibilidad en clase

# Distribución realista de tipos de transacción
tipos = ["PAYMENT", "TRANSFER", "CASH_OUT", "CASH_IN", "DEBIT"]
probabilidades = [0.35, 0.15, 0.30, 0.18, 0.02]

# Construimos un DataFrame de Pandas con datos aleatorios pero coherentes
df_pd = pd.DataFrame({
"step": np.random.randint(1, 100, n_rows),
"type": np.random.choice(tipos, n_rows, p=probabilidades),
"amount": np.round(np.random.exponential(scale=100000, size=n_rows), 2),
"nameOrig": ["C" + str(i) for i in np.random.randint(10000, 99999, n_rows)],
"oldbalanceOrg": np.round(np.random.exponential(scale=150000, size=n_rows), 2),
"nameDest": ["C" + str(i) for i in np.random.randint(10000, 99999, n_rows)],
"oldbalanceDest": np.round(np.random.exponential(scale=200000, size=n_rows), 2),
})

# Lógica básica de alteración de saldos
df_pd["newbalanceOrig"] = np.maximum(df_pd["oldbalanceOrg"] - df_pd["amount"], 0)
df_pd["newbalanceDest"] = df_pd["oldbalanceDest"] + df_pd["amount"]

# Inyectar Fraude (Imbalanceado): ~1% de fraude, concentrado en transferencias grandes
df_pd["isFraud"] = 0
mask_fraude = (df_pd["type"].isin(["TRANSFER", "CASH_OUT"])) & (df_pd["amount"] > 150000) & (np.random.rand(n_rows) < 0.05)
df_pd.loc[mask_fraude, "isFraud"] = 1

# Guardamos el dataset en el disco local de Colab
df_pd.to_csv("paysim_sample.csv", index=False)
print("¡Archivo 'paysim_sample.csv' creado con éxito!")

Generando archivo CSV sintético de 50,000 transacciones (esto tomará unos segundos)...
¡Archivo 'paysim_sample.csv' creado con éxito!


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("SparkML").getOrCreate()

# Cargamos el CSV generado previamente en un DataFrame de Spark
df_spark = spark.read.csv("paysim_sample.csv", header=True, inferSchema=True)
df_spark.printSchema()
df_spark.show(5)

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)

+----+-------+---------+--------+-------------+--------+--------------+------------------+--------------+-------+
|step|   type|   amount|nameOrig|oldbalanceOrg|nameDest|oldbalanceDest|    newbalanceOrig|newbalanceDest|isFraud|
+----+-------+---------+--------+-------------+--------+--------------+------------------+--------------+-------+
|  52|PAYMENT| 75260.18|  C22062|    302563.46|  C47999|     159891.14|227303.28000000003|     235151.32|      0|
|  93|PAYMENT|123158.12|  C26475|     54678.29|  C11061|     495862.32|               0.0|     619020.44|      0|
|  15|PAYM

**PySpark Machine Learning**

In [4]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

## Construcción y evaluación del modelo de detección de fraude con Spark ML

Esta sección construye un flujo completo de Machine Learning distribuido con `PySpark ML`. El objetivo es predecir `isFraud`, una variable binaria:

- `0`: la transacción no está marcada como fraude.
- `1`: la transacción está marcada como fraude.

Por tanto, este problema es de **clasificación binaria**, no de regresión ni de pronóstico temporal. El modelo recibe información de cada transacción y devuelve una clase predicha y sus probabilidades.

### 1. Variables de entrada y variable objetivo

El `DataFrame` `df_spark` contiene una fila por transacción. La columna `isFraud` es la **etiqueta** (`label`) que queremos predecir. Las columnas utilizadas como entrada son:

- `type`: tipo categórico de operación, por ejemplo `PAYMENT`, `TRANSFER` o `CASH_OUT`.
- `amount`: importe de la transacción.
- `oldbalanceOrg` y `newbalanceOrig`: saldo del origen antes y después.
- `oldbalanceDest` y `newbalanceDest`: saldo del destino antes y después.

La etiqueta no debe incluirse en `features`, porque eso permitiría al modelo conocer la respuesta durante el entrenamiento.

### 2. Conversión de variables categóricas

Los algoritmos de Spark ML trabajan con vectores numéricos, por lo que `type` debe transformarse:

1. `StringIndexer` aprende un diccionario y convierte cada categoría en un índice numérico (`typeIndex`).
2. `OneHotEncoder` transforma ese índice en un vector disperso (`typeVec`). Esto evita tratar, por ejemplo, `PAYMENT=0` y `TRANSFER=1` como si existiera una relación numérica entre ambas categorías.

El diccionario se aprende al ejecutar `pipeline.fit(train_data)`. Durante `transform(test_data)`, el mismo diccionario aprendido con entrenamiento se reutiliza.

### 3. Ensamblado del vector de características

`VectorAssembler` combina `typeVec` y las variables numéricas en una única columna `features`. Esta es la interfaz estándar que espera `RandomForestClassifier`:

```python
features = [variables_categoricas_codificadas + variables_numericas]
label = isFraud
```

Conceptualmente, cada fila pasa a tener la forma:

```text
(features, label) -> (vector numerico de la transaccion, 0 o 1)
```

### 4. División entre entrenamiento y prueba

```python
train_data, test_data = df_spark.randomSplit([0.8, 0.2], seed=42)
```

El 80 % se usa para aprender y el 20 % restante para medir el comportamiento sobre datos no vistos. `seed=42` hace que la división sea reproducible.

En fraude, esta división aleatoria es una primera aproximación didáctica. En producción conviene comprobar que no hay transacciones del futuro en el entrenamiento. Si existe una dimensión temporal, suele ser más realista entrenar con fechas antiguas y probar con fechas posteriores. También puede ser necesario dividir por cliente para evitar que el modelo memorice patrones del mismo cliente en ambos conjuntos.

### 5. Entrenamiento del Random Forest

```python
rf = RandomForestClassifier(
    labelCol="isFraud",
    featuresCol="features",
    numTrees=20
)
```

Un Random Forest construye muchos árboles de decisión usando subconjuntos de filas y variables. Cada árbol genera una predicción y el bosque las combina. En este ejemplo:

- `labelCol` indica cuál es la respuesta correcta.
- `featuresCol` indica dónde está el vector de entrada.
- `numTrees=20` controla cuántos árboles se entrenan. Más árboles suelen estabilizar el resultado, pero aumentan coste y tiempo.

El modelo aprende reglas como combinaciones de tipo de operación, importe y cambios de saldo. Estas reglas no son garantías causales: describen patrones presentes en los datos de entrenamiento.

### 6. Por qué se usa un `Pipeline`

```python
pipeline = Pipeline(stages=[indexer, encoder, assembler, rf])
model = pipeline.fit(train_data)
```

El pipeline encadena todo el flujo:

```text
texto -> índice -> one-hot -> vector features -> clasificador
```

Esto evita aplicar transformaciones manualmente de forma distinta en entrenamiento y prueba. `fit` aprende los parámetros necesarios y devuelve un `PipelineModel` ya entrenado. Para predecir, basta con llamar a `transform` sobre nuevos datos.

### 7. Predicciones y significado de las columnas

```python
predictions = model.transform(test_data)
predictions.select("isFraud", "prediction", "probability").show(5)
```

Spark añade normalmente estas columnas:

- `rawPrediction`: puntuaciones internas del clasificador.
- `probability`: estimación para las clases `[0, 1]`. En un Random Forest está relacionada con los votos de los árboles, pero no debe interpretarse automáticamente como una probabilidad perfectamente calibrada.
- `prediction`: clase final, normalmente `0.0` o `1.0`, según la comparación de las puntuaciones con el umbral configurado.

Para una decisión operativa no basta con mirar `prediction`. Un banco puede preferir revisar manualmente más operaciones sospechosas y aceptar más falsas alarmas. En ese caso se debe estudiar `probability` y ajustar el umbral según el coste de los falsos positivos y falsos negativos.

### 8. Evaluación con AUC

```python
evaluator = BinaryClassificationEvaluator(labelCol="isFraud")
auc = evaluator.evaluate(predictions)
```

El AUC-ROC mide la capacidad del modelo para ordenar los fraudes por encima de las operaciones legítimas en distintos umbrales:

- `0.5`: comportamiento equivalente al azar.
- cercano a `1.0`: buena separación entre clases.
- menor que `0.5`: las puntuaciones están invertidas o existe un problema en los datos.

El AUC no indica por sí solo cuántas alertas serán realmente fraude. Como el fraude suele ser una clase minoritaria, también conviene calcular matriz de confusión, `precision`, `recall`, `F1` y especialmente el área bajo la curva precisión-recall. En negocio, `recall` mide cuánto fraude detectamos y `precision` qué proporción de las alertas era fraude.

### 9. Riesgos y mejoras para evolucionar el modelo

Este notebook es una demostración funcional, pero un sistema real debería revisar al menos estos puntos:

1. **Desbalanceo**: hay muchas más operaciones legítimas que fraudulentas. La exactitud (`accuracy`) puede ser engañosa. Hay que medir el rendimiento de la clase `1`, ajustar pesos de clase o usar estrategias de muestreo.
2. **Fuga de información**: todas las variables deben estar disponibles en el instante en que se toma la decisión. Una variable calculada después de confirmar el fraude no puede utilizarse como entrada.
3. **Partición temporal**: si los datos representan operaciones a lo largo del tiempo, probar con registros posteriores aproxima mejor el uso real que `randomSplit`.
4. **Calidad de los identificadores**: `nameOrig` y `nameDest` son identificadores de alta cardinalidad. Incluirlos directamente puede producir sobreajuste; es preferible crear variables agregadas, como número de operaciones previas o importe medio por cliente.
5. **Reproducibilidad y seguimiento**: conviene guardar el pipeline, las versiones de datos, los parámetros, las métricas y la fecha de entrenamiento.
6. **Monitorización**: después del despliegue hay que vigilar cambios en la distribución de importes, tipos de operación y tasa real de fraude, además del rendimiento cuando lleguen etiquetas verificadas.

### 10. Parámetros importantes de `RandomForestClassifier`

Los parámetros pueden consultarse con:

```python
print(rf.explainParams())
```

En la salida, `default` es el valor que Spark usaría si no lo configuramos y `current` es el valor que tiene este objeto. El valor actual solo aparece si lo hemos asignado explícitamente o si Spark lo distingue del valor por defecto.

#### Estructura del bosque y muestreo

- **`numTrees`**: número de árboles del bosque. Su valor por defecto es `20` y en este notebook también se usa `20`. Aumentarlo suele reducir la variabilidad del modelo, pero aumenta tiempo, memoria y tamaño del modelo guardado. No significa número de ramas.
- **`bootstrap`**: indica si cada árbol se entrena con una muestra obtenida con reemplazo. Por defecto es `True`, que es el comportamiento clásico de un Random Forest. El muestreo con reemplazo hace que cada árbol vea una combinación ligeramente distinta de registros.
- **`subsamplingRate`**: fracción del conjunto de entrenamiento utilizada para aprender cada árbol. Por defecto es `1.0`. Junto con `bootstrap=True`, cada árbol recibe una muestra de tamaño equivalente al conjunto de entrenamiento, aunque algunos registros pueden repetirse y otros quedar fuera.
- **`featureSubsetStrategy`**: número de características candidatas en cada división de cada árbol. Por defecto es `auto`. En un bosque de clasificación con más de un árbol, `auto` elige `sqrt`, es decir, aproximadamente la raíz cuadrada del número de características. Las opciones son `all`, `onethird`, `sqrt`, `log2` o un número/fracción válido. Usar subconjuntos de variables reduce la correlación entre árboles.

Por ejemplo, si el vector `features` tiene 9 componentes y se usa `sqrt`, cada nodo considera aproximadamente `sqrt(9) = 3` variables candidatas. No significa que el árbol completo use solo tres variables: la selección se realiza en cada división.

#### Complejidad y control del árbol

- **`maxDepth`**: profundidad máxima de cada árbol. Por defecto es `5`. Una profundidad 0 produce una sola hoja; una profundidad 1 produce un nodo de decisión y hasta dos hojas. Mayor profundidad permite reglas más complejas, pero puede sobreajustar y aumenta el coste.
- **`maxBins`**: número máximo de intervalos para discretizar variables continuas. Por defecto es `32`. También debe ser, como mínimo, igual al número de categorías de cualquier variable categórica. Aumentarlo permite divisiones más precisas, pero consume más memoria.
- **`minInstancesPerNode`**: mínimo de observaciones que debe tener cada hijo después de una división. Por defecto es `1`. Aumentarlo evita crear reglas basadas en muy pocos casos.
- **`minInfoGain`**: ganancia mínima de información necesaria para aceptar una división. Por defecto es `0.0`. Un valor mayor hace el árbol más conservador y puede reducir sobreajuste.
- **`minWeightFractionPerNode`**: fracción mínima del peso total que debe conservar cada hijo. Por defecto es `0.0`. Es especialmente relevante si se utiliza `weightCol` para dar más importancia a los fraudes.
- **`impurity`**: criterio para medir la calidad de una división. En clasificación admite `gini` y `entropy`; por defecto es `gini`. Ambos buscan separar mejor las clases, aunque calculan la impureza de forma diferente.

#### Columnas de entrada y salida

- **`featuresCol`**: nombre de la columna con el vector de entrada. Por defecto es `features` y en este notebook se mantiene ese valor.
- **`labelCol`**: nombre de la columna objetivo. Por defecto es `label`, pero aquí se cambia a `isFraud`.
- **`predictionCol`**: nombre de la columna con la clase final. Por defecto es `prediction`.
- **`probabilityCol`**: nombre de la columna con las puntuaciones por clase. Por defecto es `probability`. En este modelo contiene normalmente un vector como `[0.98, 0.02]`.
- **`rawPredictionCol`**: nombre de la columna con las puntuaciones internas, también llamadas confianzas. Por defecto es `rawPrediction`.
- **`leafCol`**: nombre opcional de la columna que guarda el índice de la hoja donde termina cada instancia en cada árbol, siguiendo un recorrido en preorden. Puede ser útil para analizar qué regiones del bosque reciben determinados casos, pero no se utiliza en este notebook si se deja vacío.

#### Rendimiento distribuido y memoria

- **`cacheNodeIds`**: por defecto es `False`. Cuando es falso, el algoritmo vuelve a recorrer los árboles en los ejecutores para determinar en qué nodo está cada instancia. Si es `True`, Spark almacena los identificadores de nodo de cada instancia y puede acelerar el entrenamiento de árboles profundos, a cambio de consumir más memoria.
- **`checkpointInterval`**: cada cuántas iteraciones se comprueba el estado de la caché de nodos. Por defecto es `10`. Un valor `-1` desactiva el checkpoint. Este ajuste solo tiene efecto si se ha configurado un directorio de checkpoint en el `SparkContext`; de lo contrario, Spark ignora el valor.
- **`maxMemoryInMB`**: memoria máxima para agregar histogramas durante el entrenamiento. Por defecto es `256 MB`. Si es demasiado baja, Spark puede dividir un solo nodo por iteración y el entrenamiento será más lento; si se aumenta demasiado, puede presionar la memoria de los ejecutores.

En la mayoría de modelos pequeños no es necesario activar `cacheNodeIds`. Puede ser útil con árboles profundos, muchos datos y un coste de entrenamiento dominado por los recorridos de nodos.

#### Reproducibilidad y clases minoritarias

- **`seed`**: semilla aleatoria. El valor por defecto es aleatorio, por lo que conviene fijarlo cuando se quiera reproducir exactamente el entrenamiento. En este notebook la división usa `seed=42`, pero el bosque también puede recibir explícitamente `seed=42`:

```python
rf = RandomForestClassifier(
    labelCol="isFraud",
    featuresCol="features",
    numTrees=20,
    seed=42
)
```

- **`weightCol`**: columna opcional con el peso de cada fila. Por defecto no está definida y todas las filas pesan `1.0`. Para fraude se puede dar más peso a la clase `1`, de forma que un falso negativo resulte más costoso para el aprendizaje. El peso no crea nuevos fraudes, pero modifica la importancia que tienen durante la construcción de los árboles.

Un ejemplo conceptual sería:

```python
from pyspark.sql.functions import when

df_spark = df_spark.withColumn(
    "classWeight",
    when(col("isFraud") == 1, 10.0).otherwise(1.0)
)

rf = RandomForestClassifier(
    labelCol="isFraud",
    featuresCol="features",
    weightCol="classWeight",
    numTrees=20,
    seed=42
)
```

El valor `10.0` no es universal: debe validarse con los datos y con el coste real de las falsas alarmas y de los fraudes no detectados.

#### Umbrales de decisión

- **`thresholds`**: permite ajustar el umbral de decisión por clase. Por defecto no está definido. Spark compara la probabilidad de cada clase con su umbral y predice la clase que maximiza `p / threshold`. En clasificación binaria, cambiar estos valores puede hacer que el modelo sea más sensible a los fraudes.

Esto separa dos conceptos que a menudo se confunden:

1. El bosque aprende una puntuación o una distribución de confianza.
2. El umbral convierte esa puntuación en `prediction=0.0` o `prediction=1.0`.

Por eso un modelo puede asignar puntuaciones útiles a los fraudes y, aun así, devolver demasiados `0.0` si el criterio de decisión es demasiado conservador. El umbral debe elegirse sobre un conjunto de validación y no directamente sobre el conjunto de prueba final.

### 11. Configuración razonable para este ejemplo

Una configuración explícita y reproducible para seguir experimentando sería:

```python
rf = RandomForestClassifier(
    labelCol="isFraud",
    featuresCol="features",
    numTrees=50,
    maxDepth=6,
    featureSubsetStrategy="sqrt",
    bootstrap=True,
    subsamplingRate=1.0,
    seed=42
)
```

No existe una combinación universalmente mejor. La evolución correcta consiste en cambiar pocos parámetros cada vez, medir `recall` y `precision` de la clase fraude, revisar la matriz de confusión y comparar los resultados en un conjunto de validación que represente el uso futuro.

La idea central del flujo es: **preparar correctamente los datos, encapsular las transformaciones, controlar la complejidad del bosque, dar suficiente importancia a la clase fraude, evaluar con métricas adecuadas al negocio y monitorizar el modelo después del despliegue**.

### 12. `weightCol`: dar más importancia a los fraudes

`weightCol` es el nombre de una columna numérica que contiene el peso de cada fila. No es una característica que el modelo utilice para decidir si una transacción es fraudulenta; es una instrucción para decirle cuánto debe influir cada fila durante el entrenamiento.

Sin `weightCol`, todas las filas tienen el mismo peso:

```text
operación normal -> peso 1.0
fraude            -> peso 1.0
```

Con una columna de pesos podemos hacer que los errores sobre la clase minoritaria tengan más influencia:

```text
operación normal -> peso 1.0
fraude            -> peso 10.0
```

Esto es útil cuando hay muchos más casos `isFraud=0` que casos `isFraud=1`. Sin pesos, un modelo puede aprender una solución aparentemente cómoda: clasificar casi todo como `0.0`. Esa solución puede tener una exactitud alta, pero un `recall` de fraude igual a cero.

#### Crear una columna de pesos

```python
from pyspark.sql.functions import col, when

df_spark = df_spark.withColumn(
    "classWeight",
    when(col("isFraud") == 1, 10.0).otherwise(1.0)
)
```

La columna debe ser numérica y estar disponible tanto en `train_data` como en el `DataFrame` que se utilice para entrenar el pipeline:

```python
train_data, test_data = df_spark.randomSplit([0.8, 0.2], seed=42)
```

`classWeight` no debe incluirse en `inputCols` de `VectorAssembler`. No es una característica predictiva; solo modifica la importancia de cada observación durante `fit`.

#### Utilizar `weightCol` en el clasificador

```python
rf_weighted = RandomForestClassifier(
    labelCol="isFraud",
    featuresCol="features",
    weightCol="classWeight",
    numTrees=50,
    maxDepth=6,
    seed=42
)

pipeline_weighted = Pipeline(
    stages=[indexer, encoder, assembler, rf_weighted]
)

model_weighted = pipeline_weighted.fit(train_data)
predictions_weighted = model_weighted.transform(test_data)
```

El modelo sigue produciendo las mismas columnas de salida:

```python
predictions_weighted.select(
    "isFraud", "prediction", "probability"
).show(10)
```

La diferencia está en cómo se construyen los árboles. Un fraude con peso `10.0` influye aproximadamente como diez filas normales con peso `1.0` al calcular las divisiones y la impureza. Esto no duplica físicamente las filas ni crea nuevos fraudes; cambia su influencia matemática.

#### Elegir los pesos

Un valor fijo es fácil de entender:

```python
when(col("isFraud") == 1, 5.0).otherwise(1.0)
```

Pero también se puede calcular una ponderación aproximada para compensar el número de ejemplos de cada clase. Una estrategia habitual para dos clases es:

$$
weight_0 = \frac{N}{2N_0}, \qquad
weight_1 = \frac{N}{2N_1}
$$

Donde `N` es el número total de filas, `N0` el número de operaciones normales y `N1` el número de fraudes. La clase minoritaria recibe un peso mayor porque tiene menos ejemplos.

El valor óptimo depende del negocio. Se puede experimentar con:

```text
peso del fraude: 1, 2, 5, 10, 20
```

y comparar siempre en el mismo conjunto de validación:

- `recall` de la clase fraude: cuántos fraudes detectamos.
- `precision` de la clase fraude: cuántas alertas son realmente fraude.
- `F1`: equilibrio entre `precision` y `recall`.
- número total de falsas alarmas.
- coste estimado de un fraude no detectado frente al coste de revisar una alerta legítima.

#### Qué cambia y qué no cambia

`weightCol` puede modificar:

- Las divisiones elegidas por los árboles.
- La importancia relativa de los fraudes en el aprendizaje.
- La cantidad de predicciones `1.0`.
- El `recall` y la `precision` de la clase fraude.

`weightCol` no garantiza que todas las transacciones fraudulentas se clasifiquen como `1.0`. El modelo puede seguir equivocándose, especialmente si las características no contienen suficiente información.

Tampoco debe confundirse con `thresholds`:

```text
weightCol  -> actúa durante el entrenamiento
thresholds -> actúa al convertir las puntuaciones en una clase final
```

Por ejemplo, se puede entrenar con pesos para que el modelo aprenda a prestar más atención al fraude y después ajustar el umbral para decidir cuántas alertas se quieren generar.

#### Precauciones importantes

1. Los pesos deben calcularse usando información del conjunto de entrenamiento, no usando la etiqueta futura de una transacción que todavía no se conocería en producción.
2. La columna de pesos no debe formar parte de `features`, porque entonces el modelo podría aprender una señal artificial relacionada directamente con la etiqueta.
3. Las métricas deben calcularse sobre datos de prueba representativos. No conviene cambiar artificialmente los pesos del conjunto de prueba para presentar mejores resultados.
4. Aumentar demasiado el peso de la clase `1` puede producir muchas falsas alarmas y reducir la `precision`.
5. Antes de usar `weightCol` en producción conviene comprobar que la versión instalada de Spark y el estimador seleccionado lo soportan.

La comparación correcta no es solo “modelo con mayor AUC”, sino qué configuración ofrece el equilibrio adecuado entre fraudes detectados y operaciones legítimas enviadas a revisión.

In [5]:
# Construir el pipeline de Machine Learning con Spark ML

indexer = StringIndexer(inputCol="type", outputCol="typeIndex")
encoder = OneHotEncoder(inputCol="typeIndex", outputCol="typeVec")

assembler = VectorAssembler(
    inputCols=["typeVec", "amount", "oldbalanceOrg", "oldbalanceDest", "newbalanceOrig", "newbalanceDest"],
    outputCol="features"
)

# Entrenamiento y evaluación del modelo (TEST)
train_data, test_data = df_spark.randomSplit([0.8, 0.2], seed=42)

# Inicializar el modelo de Machine Learning
rf = RandomForestClassifier(labelCol="isFraud", featuresCol="features", numTrees=20)
print(rf.explainParams())

pipeline = Pipeline(stages=[indexer, encoder, assembler, rf])

# Entrenar el modelo
model = pipeline.fit(train_data)

# Hacer predicciones
predictions = model.transform(test_data)
predictions.select("isFraud", "prediction", "probability").show(10)

# Evaluar el modelo
evaluator = BinaryClassificationEvaluator(labelCol="isFraud")
auc = evaluator.evaluate(predictions)
print(f"AUC: {auc}")

# Analizar dentro de todo el conjunto de predicciones
predictions.groupBy("isFraud", "prediction").count().show()

# Guardamos el modelo en un fichero
model.write().overwrite().save("modelo_fraude_spark")

bootstrap: Whether bootstrap samples are used when building trees. (default: True)
cacheNodeIds: If false, the algorithm will pass trees to executors to match instances with nodes. If true, the algorithm will cache node IDs for each instance. Caching can speed up training of deeper trees. Users can set how often should the cache be checkpointed or disable it by setting checkpointInterval. (default: False)
checkpointInterval: set checkpoint interval (>= 1) or disable checkpoint (-1). E.g. 10 means that the cache will get checkpointed every 10 iterations. Note: this setting will be ignored if the checkpoint directory is not set in the SparkContext. (default: 10)
featureSubsetStrategy: The number of features to consider for splits at each tree node. Supported options: 'auto' (choose automatically for task: If numTrees == 1, set to 'all'. If numTrees > 1 (forest), set to 'sqrt' for classification and to 'onethird' for regression), 'all' (use all features), 'onethird' (use 1/3 of the featur